# Linear Regression
**Course:** Foundations of Machine Learning  
**Instructor:** Sayan CHAKI, LIRIS (UMR 5205 CNRS), École Centrale de Lyon, Université Lumière Lyon 2, INSA Lyon

**Lab 1.** Least squares from scratch (normal equations, GD, SGD), polynomial features and overfitting, ridge and lasso, evaluation on real data.

> Run in Google Colab: *Runtime → Run all*. All datasets ship with scikit-learn, so no download is needed.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

## 1. A synthetic 1D problem
We generate $y = 2 + 3x + \varepsilon$, $\varepsilon \sim \mathcal N(0, 1)$, and fit it three ways: normal equations, `lstsq`, and gradient descent.

In [ ]:
n = 100
x = np.random.uniform(-2, 2, size=n)
y = 2 + 3 * x + np.random.normal(0, 1, size=n)

X = np.c_[np.ones(n), x]          # design matrix with intercept column
plt.scatter(x, y, s=15)
plt.xlabel("x"); plt.ylabel("y"); plt.title("Synthetic data")
plt.show()

### 1.1 Normal equations
$\hat{\mathbf w} = (\mathbf X^\top \mathbf X)^{-1}\mathbf X^\top \mathbf y$. We solve the linear system rather than inverting.

In [ ]:
w_normal = np.linalg.solve(X.T @ X, X.T @ y)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("normal equations:", w_normal)
print("lstsq (SVD)     :", w_lstsq)

### 1.2 Gradient descent
$\nabla J(\mathbf w) = -\frac1n \mathbf X^\top(\mathbf y - \mathbf X\mathbf w)$. We track the loss to check convergence.

In [ ]:
def mse(X, y, w):
    r = y - X @ w
    return 0.5 * np.mean(r ** 2)

def gradient_descent(X, y, lr=0.1, epochs=200):
    w = np.zeros(X.shape[1])
    history = []
    for _ in range(epochs):
        grad = -X.T @ (y - X @ w) / len(y)
        w -= lr * grad
        history.append(mse(X, y, w))
    return w, history

for lr in [0.01, 0.1, 0.5]:
    w_gd, hist = gradient_descent(X, y, lr=lr)
    plt.plot(hist, label=f"lr={lr}")
    print(f"lr={lr}: w={w_gd}")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

### 1.3 Stochastic (mini-batch) gradient descent

In [ ]:
def sgd(X, y, lr=0.05, epochs=30, batch=10, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1]); history = []
    for _ in range(epochs):
        idx = rng.permutation(len(y))
        for s in range(0, len(y), batch):
            b = idx[s:s + batch]
            w -= lr * (-X[b].T @ (y[b] - X[b] @ w) / len(b))
        history.append(mse(X, y, w))
    return w, history

w_sgd, hist_sgd = sgd(X, y)
print("SGD:", w_sgd)
xs = np.linspace(-2, 2, 50)
plt.scatter(x, y, s=15, alpha=0.6)
plt.plot(xs, w_normal[0] + w_normal[1] * xs, "r", lw=2, label="closed form")
plt.plot(xs, w_sgd[0] + w_sgd[1] * xs, "g--", lw=2, label="SGD")
plt.legend(); plt.show()

### 1.4 Effect of feature scaling on conditioning
If one feature is on a much larger scale, $\mathbf X^\top\mathbf X$ is ill-conditioned and GD needs a tiny learning rate.

In [ ]:
X_bad = np.c_[np.ones(n), 100 * x]
print("condition number (raw)   :", np.linalg.cond(X_bad.T @ X_bad))
print("condition number (scaled):", np.linalg.cond(X.T @ X))

## 2. Probabilistic view: residuals and noise estimate
Under Gaussian noise, $\hat\sigma^2 = \frac1n \|\mathbf y - \mathbf X\hat{\mathbf w}\|^2$.

In [ ]:
res = y - X @ w_normal
print("estimated sigma:", res.std())
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X @ w_normal, res, s=12); ax[0].axhline(0, c="k")
ax[0].set_xlabel("fitted"); ax[0].set_ylabel("residual"); ax[0].set_title("Residuals vs fitted")
from scipy import stats
stats.probplot(res, dist="norm", plot=ax[1]); ax[1].set_title("Q-Q plot")
plt.tight_layout(); plt.show()

## 3. Polynomial features, overfitting and model selection
True function: $f(x)=\sin(2\pi x)$ with noise. We vary the degree $p$ and compare training and validation error.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(1)
xp = np.sort(rng.uniform(0, 1, 30))
yp = np.sin(2 * np.pi * xp) + rng.normal(0, 0.25, xp.size)
xp_val = rng.uniform(0, 1, 200)
yp_val = np.sin(2 * np.pi * xp_val) + rng.normal(0, 0.25, 200)
grid = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, p in zip(axes, [1, 3, 15]):
    m = make_pipeline(PolynomialFeatures(p), StandardScaler(), LinearRegression()).fit(xp[:, None], yp)
    ax.scatter(xp, yp, s=15); ax.plot(grid, np.sin(2*np.pi*grid), "k--", alpha=.5)
    ax.plot(grid, m.predict(grid[:, None]), "r"); ax.set_ylim(-2, 2); ax.set_title(f"degree {p}")
plt.show()

In [ ]:
degrees = range(1, 16)
tr_err, va_err = [], []
for p in degrees:
    m = make_pipeline(PolynomialFeatures(p), StandardScaler(), LinearRegression()).fit(xp[:, None], yp)
    tr_err.append(mean_squared_error(yp, m.predict(xp[:, None])))
    va_err.append(mean_squared_error(yp_val, m.predict(xp_val[:, None])))
plt.plot(degrees, tr_err, "o-", label="train")
plt.plot(degrees, va_err, "o-", label="validation")
plt.yscale("log"); plt.xlabel("degree"); plt.ylabel("MSE"); plt.legend()
plt.title("Bias-variance trade-off"); plt.show()
print("best degree:", degrees[int(np.argmin(va_err))])

## 4. Ridge regression from scratch
$\hat{\mathbf w}_{\text{ridge}} = (\mathbf X^\top\mathbf X + \lambda \mathbf I)^{-1}\mathbf X^\top\mathbf y$, without penalising the intercept (we centre the data instead).

In [ ]:
def ridge_fit(X, y, lam):
    mu_x, mu_y = X.mean(0), y.mean()
    Xc, yc = X - mu_x, y - mu_y
    w = np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ yc)
    b = mu_y - mu_x @ w
    return w, b

feat = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler()).fit(xp[:, None])
Phi, Phi_grid = feat.transform(xp[:, None]), feat.transform(grid[:, None])

plt.scatter(xp, yp, s=15, c="k")
for lam in [1e-6, 1e-2, 1, 10]:
    w, b = ridge_fit(Phi, yp, lam)
    plt.plot(grid, Phi_grid @ w + b, label=f"lambda={lam}")
plt.ylim(-2, 2); plt.legend(); plt.title("Ridge, degree 15"); plt.show()

## 5. Real data: the diabetes dataset
442 patients, 10 baseline features, target = disease progression after one year.

In [ ]:
from sklearn.datasets import load_diabetes
data = load_diabetes()
Xd, yd = data.data, data.target
print(Xd.shape, data.feature_names)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=0)

models = {
    "OLS":   make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)),
}
for name, m in models.items():
    m.fit(Xd_tr, yd_tr)
    pred = m.predict(Xd_te)
    print(f"{name:6s} RMSE={mean_squared_error(yd_te, pred)**0.5:6.2f}  R2={r2_score(yd_te, pred):.3f}")

### 5.1 Regularisation paths: ridge shrinks, lasso selects

In [ ]:
Xs = StandardScaler().fit_transform(Xd_tr)
alphas_r = np.logspace(-2, 4, 60)
alphas_l = np.logspace(-2, 2, 60)
coef_r = np.array([Ridge(alpha=a).fit(Xs, yd_tr).coef_ for a in alphas_r])
coef_l = np.array([Lasso(alpha=a, max_iter=10000).fit(Xs, yd_tr).coef_ for a in alphas_l])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(alphas_r, coef_r); ax[0].set_xscale("log"); ax[0].set_title("Ridge path"); ax[0].set_xlabel("alpha")
ax[1].plot(alphas_l, coef_l); ax[1].set_xscale("log"); ax[1].set_title("Lasso path"); ax[1].set_xlabel("alpha")
ax[1].legend(data.feature_names, fontsize=7, loc="upper right")
plt.show()

lasso = models["Lasso"].named_steps["lassocv"]
print("Lasso alpha:", lasso.alpha_)
print("Selected features:", [f for f, c in zip(data.feature_names, lasso.coef_) if abs(c) > 1e-8])

### 5.2 Learning curve

In [ ]:
from sklearn.model_selection import learning_curve
sizes, tr, va = learning_curve(models["Ridge"], Xd, yd, cv=5, scoring="neg_mean_squared_error",
                               train_sizes=np.linspace(0.1, 1.0, 8))
plt.plot(sizes, -tr.mean(1), "o-", label="train")
plt.plot(sizes, -va.mean(1), "o-", label="CV")
plt.xlabel("training set size"); plt.ylabel("MSE"); plt.legend(); plt.show()

## 6. Exercises
1. Implement **lasso with coordinate descent** using the soft-thresholding operator $S_\lambda(z)=\mathrm{sign}(z)\max(|z|-\lambda,0)$ and compare with `sklearn.linear_model.Lasso` on the diabetes data.
2. Verify numerically that $(\mathbf X^\top\mathbf X+\lambda\mathbf I)^{-1}\mathbf X^\top\mathbf y = \mathbf X^\top(\mathbf X\mathbf X^\top+\lambda\mathbf I)^{-1}\mathbf y$.
3. Add an outlier with a huge target to the synthetic data. Compare OLS with `HuberRegressor`.
4. Use `RidgeCV` with `PolynomialFeatures` inside a pipeline and tune the degree with `GridSearchCV`.

In [ ]:
# Exercise 1: your code here
def soft_threshold(z, lam):
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

def lasso_cd(X, y, lam, n_iter=200):
    n, d = X.shape
    w = np.zeros(d)
    # TODO: cycle over coordinates j, compute partial residual, update w[j]
    return w